# WP 10 · Exploring Colour Space
*Live lab — Unit II: UAV Data Collection & Processing Methods*

A colour space is a coordinate system for colour. RGB is how sensors capture; HSV separates *what colour* from *how bright* - which is why segmentation and tracking live there. In OpenCV, **H is 0-179**.

**Supporting data:** `aerial_scene.jpg` (download it from the button next to this snippet in the playground, then upload when the notebook asks).

---
**How to use:** `Runtime ▸ Run all`, or run each cell top to bottom. The original teaching snippet is reproduced verbatim below; only GUI-only calls (`cv2.imshow`, `cv2.waitKey`, `cv2.destroyAllWindows`) are adapted, because Colab has no display window.

In [ ]:
# === Setup (run me first) ===================================================
# OpenCV, NumPy and Matplotlib are already installed in Google Colab.
# If you run locally and cv2 is missing, uncomment the next line:
# !pip install opencv-python-headless matplotlib

import cv2, numpy as np, os
import matplotlib.pyplot as plt
print("OpenCV", cv2.__version__)

def show(*imgs, titles=None, cmap=None, figsize=(13, 5)):
    """Display 1..N images inline. BGR images are auto-converted to RGB.
    (Colab has no window server, so cv2.imshow() cannot be used.)"""
    titles = titles or [""] * len(imgs)
    plt.figure(figsize=figsize)
    for i, im in enumerate(imgs):
        ax = plt.subplot(1, len(imgs), i + 1)
        if im.ndim == 2:
            ax.imshow(im, cmap=cmap or "gray", vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(titles[i], fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()


In [ ]:
# --- Stand-in scene generator (used only if you don't upload the data file) ---
def make_scene(w=768, h=512, seed=7):
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), (78, 120, 96), np.uint8)
    yy, xx = np.mgrid[0:h, 0:w]
    img = np.clip(img + (16*np.sin(xx/130) + 10*np.cos(yy/90))[..., None], 0, 255).astype(np.uint8)
    crop = img.copy()
    for k in range(-h, w, 9):
        cv2.line(crop, (k, 0), (k + h, h), (60, 165, 90), 2, cv2.LINE_AA)
    m = np.zeros((h, w), np.uint8); cv2.rectangle(m, (0, 150), (470, h), 255, -1)
    img[m == 255] = crop[m == 255]
    cv2.ellipse(img, (628, 120), (95, 62), 18, 0, 360, (150, 92, 40), -1, cv2.LINE_AA)
    pts = np.array([[0,470],[180,430],[330,360],[430,250],[520,170],[640,90],[w,40]], np.int32)
    cv2.polylines(img, [pts], False, (70, 72, 78), 26, cv2.LINE_AA)
    for cx, cy, bw, bh, col, a in [(150,250,84,60,(205,205,210),8),(250,300,70,52,(120,150,225),-6),
                                   (360,190,60,44,(225,225,230),20),(120,360,56,40,(150,175,235),4)]:
        r = cv2.boxPoints(((cx,cy),(bw,bh),a)).astype(np.int32)
        cv2.fillConvexPoly(img, r, col, cv2.LINE_AA); cv2.polylines(img, [r], True, (40,40,45), 2, cv2.LINE_AA)
    for cx, cy, rr in [(60,120,16),(95,175,13),(300,110,15),(430,430,18),(500,470,14),(700,300,17),(610,380,13),(250,460,15)]:
        cv2.circle(img, (cx, cy), rr, (40, 95, 45), -1, cv2.LINE_AA)
    for cx, cy, col, a in [(300,380,(250,250,250),-32),(470,205,(60,60,235),40)]:
        r = cv2.boxPoints(((cx,cy),(26,12),a)).astype(np.int32); cv2.fillConvexPoly(img, r, col, cv2.LINE_AA)
    return img

def get_image(name, gray=False):
    """Load `name` if present; else offer a Colab upload; else auto-generate."""
    flag = cv2.IMREAD_UNCHANGED if name.lower().endswith(".png") else cv2.IMREAD_COLOR
    if os.path.exists(name):
        im = cv2.imread(name, flag)
        if im is not None:
            print("Loaded", name)
            return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    try:
        from google.colab import files
        print(f"Upload '{name}' from the data pack (or press Cancel to auto-generate).")
        up = files.upload()
        for fn in up:
            im = cv2.imread(fn, flag)
            if im is not None:
                print("Loaded", fn)
                return cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) if gray else im
    except Exception:
        pass
    print("Auto-generating a stand-in scene.")
    s = make_scene()
    return cv2.cvtColor(s, cv2.COLOR_BGR2GRAY) if gray else s


### The snippet, exactly as shown in the playground
```python
hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# segment vegetation by hue (see WP 14):
mask = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))
```

In [ ]:
img = get_image("aerial_scene.jpg")

hsv  = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
lab  = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# segment vegetation by hue (green field / trees):
mask = cv2.inRange(hsv, (35, 40, 40), (85, 255, 255))

veg = cv2.bitwise_and(img, img, mask=mask)   # keep only vegetation pixels
show(img, mask, veg,
     titles=["Original (BGR)", "HSV vegetation mask", "Vegetation only"])
print(f"Vegetation covers {100*mask.mean()/255:.1f}% of the frame")

In [ ]:
# --- Split the HSV channels so you can *see* what H, S, V each encode --------
H, S, V = cv2.split(hsv)
show(H, S, V, titles=["Hue (0-179)", "Saturation", "Value / brightness"])
# Notice: drop V and the colour darkens but its Hue stays put - that hue
# independence is why an HSV mask survives a passing cloud where RGB fails.